In [1]:
import os  # Lets us set the Java location before Spark starts.
import shutil  # Finds Ubuntu's Java command.
from pathlib import Path  # Helps us build filesystem paths.
from pyspark.sql import SparkSession  # Creates our Spark session.
from delta import configure_spark_with_delta_pip  # Adds Delta's Java package.

java_command = shutil.which("java")  # Find Java in the WSL environment.
if java_command is None:  # Check that Java was found.
    raise RuntimeError("Java was not found in WSL")  # Stop with a clear message if it was not.

java_home = Path(java_command).resolve().parent.parent  # Find the Java installation folder.
os.environ["JAVA_HOME"] = str(java_home)  # Point Spark to Ubuntu's Java.

project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Locate our project.

builder = (  # Define Spark's settings before it starts.
    SparkSession.builder
    .master("local[2]")  # Use two processing threads on this computer.
    .appName("RetailSilverSuppliers")  # Name this Silver job.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta features.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")  # Enable Delta tables.
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()  # Start Spark with Delta support.

print("Project found:", project_root.is_dir())  # Confirm the project path works.
print("Spark version:", spark.version)  # Confirm Spark started.

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/09 18:30:38 WARN Utils: Your hostname, USMAN-ALI, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/09 18:30:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/usmanali611b/.venvs/retail-lakehouse/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/usmanali611b/.ivy2.5.2/cache
The jars for the packages stored in: /home/usmanali611b/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5c40e7b5-8452-49f3-8251-7eb4c88f5e1b;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.1

Project found: True
Spark version: 4.1.1


In [2]:
bronze_suppliers_path = project_root / "lakehouse" / "bronze" / "data" / "suppliers"  # Locate the suppliers Bronze table.
suppliers_bronze_df = spark.read.format("delta").load(str(bronze_suppliers_path))  # Read the saved Delta table.

print("Bronze supplier rows:", suppliers_bronze_df.count())  # Check how many suppliers arrived.
suppliers_bronze_df.printSchema()  # See the column names and types.
suppliers_bronze_df.show(5, truncate=False)  # Inspect five complete rows.

26/10/09 18:31:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/10/09 18:35:44 WARN NettyRpcEnv: Ignored failure: java.util.concurrent.TimeoutException: Cannot receive any reply from 10.255.255.254:39133 in 10000 milliseconds
                                                                                

Bronze supplier rows: 18
root
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- lead_time_days: string (nullable = true)
 |-- active_flag: string (nullable = true)



[Stage 13:>                                                         (0 + 1) / 1]

+-----------+-----------------------+-------------+----------------------+---------------+----------+--------------+-----------+
|supplier_id|supplier_name          |contact_name |email                 |phone          |city      |lead_time_days|active_flag|
+-----------+-----------------------+-------------+----------------------+---------------+----------+--------------+-----------+
|SUP001     |Atlas Foods            |Noah Robinson|contact001@example.com|+92-300-1000001|Multan    |4             |True       |
|SUP002     |Blue River Trading     |Sophia Patel |contact002@example.com|+92-300-1000002|Faisalabad|4             |True       |
|SUP003     |Cedar Household        |Aarav Brown  |contact003@example.com|+92-300-1000003|Faisalabad|2             |True       |
|SUP004     |Crescent Consumer Goods|William Khan |contact004@example.com|+92-300-1000004|Lahore    |2             |True       |
|SUP005     |Evergreen Supplies     |Emma Ahmed   |contact005@example.com|+92-300-1000005|Karachi

In [3]:
from pyspark.sql import functions as F  # Import Spark's column functions.

missing_id_count = suppliers_bronze_df.filter(  # Find suppliers without a usable ID.
    F.col("supplier_id").isNull()  # Include null IDs.
    | (F.trim(F.col("supplier_id")) == "")  # Include empty or spaces-only IDs.
).count()  # Count those rows.

suppliers_with_clean_id_df = suppliers_bronze_df.withColumn(  # Add an ID with outside spaces removed.
    "clean_supplier_id", F.trim(F.col("supplier_id"))  # Use the form Silver would store.
)

duplicate_id_groups = (  # Count IDs that would collide after trimming.
    suppliers_with_clean_id_df
    .groupBy("clean_supplier_id")  # Group rows with the same cleaned ID.
    .count()  # Count rows in each group.
    .filter(F.col("count") > 1)  # Keep groups containing duplicates.
    .count()  # Count those groups.
)

missing_name_count = suppliers_bronze_df.filter(  # Find suppliers without a usable name.
    F.col("supplier_name").isNull()  # Include null names.
    | (F.trim(F.col("supplier_name")) == "")  # Include empty or spaces-only names.
).count()  # Count those rows.

print("Missing supplier IDs:", missing_id_count)  # Show the key check.
print("Duplicate cleaned supplier IDs:", duplicate_id_groups)  # Show the uniqueness check.
print("Missing supplier names:", missing_name_count)  # Show the name check.

suppliers_bronze_df.select("lead_time_days").distinct().orderBy("lead_time_days").show()  # Inspect lead-time text values.
suppliers_bronze_df.groupBy("active_flag").count().orderBy("active_flag").show()  # Inspect flag text values.

Missing supplier IDs: 0
Duplicate cleaned supplier IDs: 0
Missing supplier names: 0
+--------------+
|lead_time_days|
+--------------+
|            10|
|             2|
|             3|
|             4|
|             5|
|             7|
+--------------+

+-----------+-----+
|active_flag|count|
+-----------+-----+
|       True|   18|
+-----------+-----+



In [6]:
def optional_text(column_name):  # Clean a text field that the source allows to be missing.
    value = F.trim(F.col(column_name))  # Remove spaces at the beginning and end.
    return F.when(value == "", F.lit(None)).otherwise(value)  # Represent blank text as null.

suppliers_clean_df = suppliers_bronze_df.select(  # Create new columns without changing Bronze.
    F.trim(F.col("supplier_id")).alias("supplier_id"),  # Clean the supplier key.
    F.trim(F.col("supplier_name")).alias("supplier_name"),  # Clean the required name.
    optional_text("contact_name").alias("contact_name"),  # Clean the optional contact.
    optional_text("email").alias("email"),  # Clean the optional email.
    optional_text("phone").alias("phone"),  # Clean the optional phone.
    optional_text("city").alias("city"),  # Clean the optional city.
    F.trim(F.col("lead_time_days")).try_cast("int").alias("lead_time_days"),  # Convert valid text to an integer.
    F.trim(F.col("active_flag")).try_cast("boolean").alias("active_flag"),  # Convert valid text to a boolean.
    F.to_json(F.struct(*[F.col(name) for name in suppliers_bronze_df.columns])).alias("source_record"),  # Preserve the original CSV row for error evidence.
)

print("Cleaned supplier rows:", suppliers_clean_df.count())  # Check that cleaning retained all rows.
suppliers_clean_df.printSchema()  # Confirm the two converted data types.
suppliers_clean_df.show(5, truncate=False)  # Inspect the converted values.

Cleaned supplier rows: 18
root
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- lead_time_days: integer (nullable = true)
 |-- active_flag: boolean (nullable = true)
 |-- source_record: string (nullable = true)

+-----------+-----------------------+-------------+----------------------+---------------+----------+--------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|supplier_id|supplier_name          |contact_name |email                 |phone          |city      |lead_time_days|active_flag|source_record                                                                                                          

In [8]:
from pyspark.sql.window import Window  # Lets us count rows with the same cleaned supplier ID.

def missing_required_text(column_name):  # Build a reusable check for required text.
    column = F.col(column_name)  # Select the named Spark column.
    return column.isNull() | (column == "")  # Flag null or empty text after trimming.

suppliers_checked_df = suppliers_clean_df.withColumn(  # Add a count for each cleaned supplier ID.
    "id_occurrences",  # Name the temporary count column.
    F.count(F.lit(1)).over(Window.partitionBy("supplier_id")),  # Count rows sharing this ID.
).withColumn(  # Add the reasons a row cannot enter Silver.
    "quality_issue",  # Store one or more issue names.
    F.concat_ws(  # Join issue names while skipping null checks.
        "; ",  # Separate multiple issues.
        F.when(missing_required_text("supplier_id"), "missing_supplier_id"),  # Require an ID.
        F.when(F.col("id_occurrences") > 1, "duplicate_supplier_id"),  # Reject duplicate cleaned IDs.
        F.when(missing_required_text("supplier_name"), "missing_supplier_name"),  # Require a name.
        F.when(F.col("email").isNotNull() & ~F.col("email").contains("@"), "email_without_at"),  # Check email only when present.
        F.when(F.col("lead_time_days").isNull(), "missing_or_invalid_lead_time"),  # Catch absent or unconvertible values.
        F.when(F.col("lead_time_days") <= 0, "lead_time_not_positive"),  # Require more than zero days.
        F.when(F.col("active_flag").isNull(), "missing_or_invalid_active_flag"),  # Require a valid boolean.
    ),
)

accepted_count = suppliers_checked_df.filter(F.col("quality_issue") == "").count()  # Count passing rows.
rejected_count = suppliers_checked_df.filter(F.col("quality_issue") != "").count()  # Count failing rows.
print("Source record available for rejection:", "source_record" in suppliers_checked_df.columns)  # Confirm the checked rows retain the original CSV values.
print("Suppliers ready for Silver:", accepted_count)  # Show accepted rows.
print("Suppliers needing rejection:", rejected_count)  # Show rejected rows.
print("All rows accounted for:", accepted_count + rejected_count == 18)  # Check the split.

Source record available for rejection: True
Suppliers ready for Silver: 18
Suppliers needing rejection: 0
All rows accounted for: True


In [9]:
silver_suppliers_df = (  # Build the DataFrame for the Silver table.
    suppliers_checked_df
    .filter(F.col("quality_issue") == "")  # Keep rows that passed every rule.
    .drop("id_occurrences", "quality_issue", "source_record")  # Keep only the eight business columns.
)

rejected_suppliers_df = (  # Build the DataFrame for error evidence.
    suppliers_checked_df
    .filter(F.col("quality_issue") != "")  # Keep rows with one or more issues.
    .drop("id_occurrences")  # Keep quality_issue and source_record for investigation.
)

print("Silver supplier candidates:", silver_suppliers_df.count())  # Expect 18.
print("Rejected supplier candidates:", rejected_suppliers_df.count())  # Expect 0.
print("Silver columns:", silver_suppliers_df.columns)  # Confirm no checking columns remain.

Silver supplier candidates: 18


Rejected supplier candidates: 0
Silver columns: ['supplier_id', 'supplier_name', 'contact_name', 'email', 'phone', 'city', 'lead_time_days', 'active_flag']


In [10]:
from uuid import uuid4  # Creates a different ID for each notebook run.

bronze_count = suppliers_bronze_df.count()  # Count the incoming Bronze rows.
silver_count = silver_suppliers_df.count()  # Count accepted rows.
rejected_count = rejected_suppliers_df.count()  # Count rejected rows.

if silver_count + rejected_count != bronze_count:  # Check every source row has an outcome.
    raise ValueError("Supplier rows were lost during Silver processing")  # Stop before writing.

run_id = str(uuid4())  # Identify this particular processing run.
saved_rejected_count = 0  # Start with no saved rejected rows.

if rejected_count > 0:  # Write error evidence only when rejected rows exist.
    error_path = project_root / "error" / "data" / "suppliers"  # Choose the ignored error-data folder.
    error_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure its parent exists.

    errors_to_save_df = (  # Add information needed to identify this run.
        rejected_suppliers_df
        .withColumn("run_id", F.lit(run_id))  # Label every rejected row with the run ID.
        .withColumn("rejected_at", F.current_timestamp())  # Record when Spark processes the row.
    )

    errors_to_save_df.write.format("delta").mode("append").save(str(error_path))  # Keep earlier error evidence.

    saved_rejected_count = (  # Count errors saved for this run only.
        spark.read.format("delta").load(str(error_path))  # Read the error Delta table.
        .filter(F.col("run_id") == run_id)  # Select this run's rejected rows.
        .count()  # Count them.
    )

    if saved_rejected_count != rejected_count:  # Verify the error write.
        raise ValueError("Saved supplier rejection count does not match")  # Stop before updating Silver.

silver_suppliers_path = project_root / "lakehouse" / "silver" / "data" / "suppliers"  # Choose the Silver table folder.
silver_suppliers_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure its parent exists.

silver_suppliers_df.write.format("delta").mode("overwrite").save(str(silver_suppliers_path))  # Save the current clean snapshot.

saved_silver_df = spark.read.format("delta").load(str(silver_suppliers_path))  # Read Silver back.
saved_silver_count = saved_silver_df.count()  # Count saved rows.

if saved_silver_count != silver_count:  # Compare saved and accepted counts.
    raise ValueError("Saved Silver supplier count does not match")  # Stop if they differ.

print("Bronze suppliers:", bronze_count)  # Show input rows.
print("Saved Silver suppliers:", saved_silver_count)  # Show clean output rows.
print("Rejected suppliers this run:", saved_rejected_count)  # Show error rows.
print("Delta log exists:", (silver_suppliers_path / "_delta_log").is_dir())  # Confirm Delta output.

Bronze suppliers: 18
Saved Silver suppliers: 18
Rejected suppliers this run: 0
Delta log exists: True
